# CineMatch — Evaluation & Benchmarks

**Held-out evaluation** using the same train/test split that XSimGCL was trained on.

1. **Loads pre-computed splits**: `train_ratings.csv` (XSimGCL training data) and `test_holdout.csv` (held-out ground truth)
2. **Baselines**: Random, MostPopular, ItemKNN, BPR-MF, LightGCN (SOTA GNN)
3. **CineMatch variants**: Content-Only (FAISS), CF-Only (XSimGCL), Full Fusion
4. **Index comparison**: TMDB BGE vs TMDB Qwen
5. **Metrics**: NDCG@K, Hit@K, MRR, ILD@K, CCDR@K, Coverage@K
6. **DPP ablation**: With vs without DPP diversity
7. **Cold vs Warm vs Power** user segment analysis

In [1]:
!pip install -q faiss-gpu-cu12 sentence-transformers recbole

from __future__ import annotations
import gc, json, math, os, sys, time, warnings, random
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import display, HTML
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
warnings.filterwarnings("ignore")

## Path Detection

In [2]:
def detect_paths() -> dict:
    def pick(base, *rel):
        for r in rel:
            p = base / r
            if p.exists(): return p
        return base / rel[0]

    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = Path("/content/drive/MyDrive/cinematch")
        rt = "Colab"
    except ImportError:
        hpc = Path("/blue/egn6933/nagabhairava.r")
        if hpc.exists():
            base = hpc; rt = "HPC"
        else:
            here = Path(".").resolve()
            for c in [here, *here.parents]:
                if (c / "Data").exists() and (c / "src").exists():
                    base = c; break
            else:
                base = Path.cwd()
            rt = "Local"

    print(f"Runtime: {rt} | Base: {base}")
    xsim = base / "outputs" / "xsimgcl"
    return {
        "base": base,
        # Pre-computed evaluation splits (from 7)XSimGCL_Train)
        "train_ratings":   xsim / "train_ratings.csv",
        "test_holdout":    xsim / "test_holdout.csv",
        "eval_split_meta": xsim / "eval_split_meta.json",
        "train_manifest":  xsim / "train_manifest.json",
        # XSimGCL embeddings
        "user_emb":    xsim / "user_embeddings.npy",
        "item_emb":    xsim / "item_embeddings.npy",
        "user_id_map": xsim / "user_id_map.json",
        "item_id_map": xsim / "item_id_map.json",
        # Catalogs
        "tmdb_catalog": pick(base, "Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv"),
        "imdb_catalog": pick(base, "outputs/imdb/imdb_movies_catalog.csv"),
        # FAISS
        "imdb_faiss":      pick(base, "outputs/imdb/imdb_movies_bge_m3_flatip.faiss"),
        "tmdb_bge_faiss":  pick(base, "outputs/tmdb/bge/tmdb_bge_m3_flatip.faiss"),
        "tmdb_qwen_faiss": pick(base, "outputs/tmdb/qwen/tmdb_qwen4b.faiss"),
        # ML metadata
        "links_csv":  pick(base, "Data/ml-32m/links.csv"),
        "movies_csv": pick(base, "Data/ml-32m/movies.csv"),
    }

P = detect_paths()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime: Colab | Base: /content/drive/MyDrive/cinematch


## Load Pre-Computed Evaluation Splits
These were created in `7)XSimGCL_Train.ipynb`

In [3]:
# Load split metadata
with open(P["eval_split_meta"]) as f:
    split_meta = json.load(f)
print("Eval Split Metadata:")
for k, v in split_meta.items():
    print(f"  {k}: {v}")

with open(P["train_manifest"]) as f:
    manifest = json.load(f)
print(f"\nXSimGCL train results: {manifest.get('results', {})}")

# Load test holdout (ground truth)
test_holdout_df = pd.read_csv(P["test_holdout"],
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"})
print(f"\nTest holdout: {len(test_holdout_df):,} rows, {test_holdout_df['userId'].nunique()} users")

#Load train ratings
print("Loading train ratings...")
t0 = time.time()
train_ratings = pd.read_csv(P["train_ratings"],
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int32"})
print(f"  Train ratings: {len(train_ratings):,} in {time.time()-t0:.1f}s")

# Build test set dict
# Group holdout by user
test_users = test_holdout_df.groupby("userId")["movieId"].apply(set).to_dict()

# Build history (all train ratings for test users)
test_user_ids = set(test_users.keys())
train_by_user = train_ratings[train_ratings["userId"].isin(test_user_ids)]

test_set = {}
for uid, ground_truth in test_users.items():
    history = train_by_user[train_by_user["userId"] == uid]
    history_mids = set(history["movieId"].tolist())
    all_watched = set(history["movieId"].tolist()) | ground_truth
    test_set[uid] = {
    "ground_truth": ground_truth,
    "history": history,
    "history_mids": history_mids,
    "all_watched": all_watched,
    }

print(f"\nTest set built: {len(test_set)} users")
print(f"  Activity distribution: {split_meta.get('activity_distribution', {})}")

# Load movie metadata
links = pd.read_csv(P["links_csv"])
links["tmdbId"] = pd.to_numeric(links["tmdbId"], errors="coerce")
ml_to_tmdb = dict(zip(links["movieId"], links["tmdbId"].astype("Int64")))
tmdb_to_ml = {int(v): int(k) for k, v in ml_to_tmdb.items() if pd.notna(v)}

movies = pd.read_csv(P["movies_csv"])
movie_genres = dict(zip(movies["movieId"], movies["genres"]))

# Language info from TMDB catalog
tmdb_cat = pd.read_csv(P["tmdb_catalog"],
    usecols=["id", "original_language"], low_memory=False)
tmdb_cat["id"] = pd.to_numeric(tmdb_cat["id"], errors="coerce")
tmdb_cat = tmdb_cat.dropna(subset=["id"]).set_index("id")
def get_lang(mid):
    tid = ml_to_tmdb.get(mid)
    if pd.notna(tid) and int(tid) in tmdb_cat.index:
        return str(tmdb_cat.loc[int(tid), "original_language"])
    return "en"


Eval Split Metadata:
  n_test_users: 979
  n_holdout_per_user: 10
  holdout_threshold: 4.0
  min_user_ratings: 20
  total_held_out: 9790
  total_train_ratings: 31990414
  total_original_ratings: 32000204
  eval_seed: 42
  activity_distribution: {'5-50': 183, '50-100': 196, '100-200': 200, '200-500': 200, '500+': 200}

XSimGCL train results: {'best_valid_score': '0.7901', 'test_result': "OrderedDict({'recall@10': 0.7277, 'recall@20': 0.8298, 'recall@50': 0.9162, 'ndcg@10': 0.7299, 'ndcg@20': 0.7564, 'ndcg@50': 0.7892, 'mrr@10': 0.8045, 'mrr@20': 0.805, 'mrr@50': 0.8051})", 'train_time_seconds': 6891.59}

Test holdout: 9,790 rows, 979 users
Loading train ratings...
  Train ratings: 31,990,414 in 15.2s

Test set built: 979 users
  Activity distribution: {'5-50': 183, '50-100': 196, '100-200': 200, '200-500': 200, '500+': 200}


## Load XSimGCL Embeddings

In [4]:
user_emb = np.load(P["user_emb"])
item_emb = np.load(P["item_emb"])
with open(P["user_id_map"]) as f:
    user_id_map = {int(k): v for k, v in json.load(f).items()}
with open(P["item_id_map"]) as f:
    item_id_map = {int(k): v for k, v in json.load(f).items()}
idx_to_movieid = {v: k for k, v in item_id_map.items()}
print(f"XSimGCL: {user_emb.shape[0]:,} users × {item_emb.shape[0]:,} items, dim={user_emb.shape[1]}")
print(f"  Trained on: {manifest['dataset'].get('train_source', 'train_ratings.csv')}")
print(f"  Users in map: {len(user_id_map):,}, Items in map: {len(item_id_map):,}")
# Verify test users are in the embedding
test_in_map = sum(1 for uid in test_set if uid in user_id_map)
print(f"  Test users with XSimGCL embeddings: {test_in_map}/{len(test_set)}")

XSimGCL: 200,807 users × 65,023 items, dim=512
  Trained on: /blue/egn6933/nagabhairava.r/outputs/xsimgcl/train_ratings.csv
  Users in map: 200,806, Items in map: 65,022
  Test users with XSimGCL embeddings: 977/979


## Load FAISS Indices & BGE-M3

In [5]:
import faiss
from sentence_transformers import SentenceTransformer

faiss_indices = {}
for name, path in [("imdb_bge", P["imdb_faiss"]),
                   ("tmdb_bge", P["tmdb_bge_faiss"]),
                   ("tmdb_qwen", P["tmdb_qwen_faiss"])]:
    if path.exists():
        print(f"Loading {name}...", end=" ")
        faiss_indices[name] = faiss.read_index(str(path))
        print(f"{faiss_indices[name].ntotal:,}")
    else:
        print(f"{name}: not found at {path}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nLoading BGE-M3 on {device}...")
bge_model = SentenceTransformer("BAAI/bge-m3", device=device)
BGE_DIM = bge_model.get_sentence_embedding_dimension() or 1024
print(f"BGE-M3: {BGE_DIM}d")

Loading imdb_bge... 737,654
Loading tmdb_bge... 1,366,255
Loading tmdb_qwen... 1,366,255

Loading BGE-M3 on cuda...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE-M3: 1024d


## Evaluation Metrics
- **NDCG@K**: Normalized Discounted Cumulative Gain — ranking quality
- **Hit@K**: Fraction of users where ≥1 ground truth item appears in top-K
- **MRR**: Mean Reciprocal Rank of first hit
- **ILD@K**: Intra-List Diversity — mean pairwise genre-based Jaccard distance
- **CCDR@K**: Cross-Cultural Diversity Ratio — fraction of non-English in top-K
- **Coverage@K**: Fraction of catalog items recommended across all users

In [6]:
def ndcg_at_k(recommended: list[int], ground_truth: set[int], k: int) -> float:
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in ground_truth:
            dcg += 1.0 / math.log2(i + 2)
    n_relevant = min(len(ground_truth), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(n_relevant))
    return dcg / idcg if idcg > 0 else 0.0

def hit_at_k(recommended: list[int], ground_truth: set[int], k: int) -> float:
    return 1.0 if any(item in ground_truth for item in recommended[:k]) else 0.0

def mrr(recommended: list[int], ground_truth: set[int]) -> float:
    for i, item in enumerate(recommended):
        if item in ground_truth:
            return 1.0 / (i + 1)
    return 0.0

def ild_at_k(recommended: list[int], k: int) -> float:
    recs = recommended[:k]
    if len(recs) < 2: return 0.0
    genre_sets = []
    for mid in recs:
        g = movie_genres.get(mid, "")
        genre_sets.append(set(g.split("|")) if g else set())
    distances = []
    for i in range(len(genre_sets)):
        for j in range(i+1, len(genre_sets)):
            union = genre_sets[i] | genre_sets[j]
            inter = genre_sets[i] & genre_sets[j]
            distances.append(1.0 - len(inter) / max(len(union), 1))
    return float(np.mean(distances)) if distances else 0.0

def ccdr_at_k(recommended: list[int], k: int) -> float:
    recs = recommended[:k]
    if not recs: return 0.0
    non_en = sum(1 for mid in recs if get_lang(mid) != "en")
    return non_en / len(recs)

def evaluate_method(method_fn, test_data: dict, k_values: list[int] = [10, 20, 30],
                    method_name: str = "Method") -> dict:
    all_recs = set()
    results = {k: {"ndcg": [], "hit": [], "mrr": [], "ild": [], "ccdr": []}
               for k in k_values}
    t0 = time.time()
    n_done = 0
    for uid, data in test_data.items():
        try:
            recommended = method_fn(uid, data)
        except Exception as e:
            continue
        if not recommended: continue
        for k in k_values:
            results[k]["ndcg"].append(ndcg_at_k(recommended, data["ground_truth"], k))
            results[k]["hit"].append(hit_at_k(recommended, data["ground_truth"], k))
            results[k]["mrr"].append(mrr(recommended, data["ground_truth"]))
            results[k]["ild"].append(ild_at_k(recommended, k))
            results[k]["ccdr"].append(ccdr_at_k(recommended, k))
        all_recs.update(recommended[:max(k_values)])
        n_done += 1
        if n_done % 100 == 0:
            print(f"  {method_name}: {n_done}/{len(test_data)}...", end="\r")
    elapsed = time.time() - t0
    total_items = len(set(movies["movieId"]))
    summary = {}
    for k in k_values:
        if results[k]["ndcg"]:
            summary[k] = {
                "NDCG": np.mean(results[k]["ndcg"]),
                "Hit": np.mean(results[k]["hit"]),
                "MRR": np.mean(results[k]["mrr"]),
                "ILD": np.mean(results[k]["ild"]),
                "CCDR": np.mean(results[k]["ccdr"]),
                "Coverage": len(all_recs) / total_items,
            }
    print(f"  {method_name}: done ({n_done} users, {elapsed:.1f}s)")
    return summary

## Baselines

### Trivial Baselines
- **Random**: Pick K random items from catalog
- **MostPopular**: Recommend K most-rated items globally (from TRAIN set only)

In [7]:
global_popularity = train_ratings.groupby("movieId").size().sort_values(ascending=False)
popular_items = global_popularity.index.tolist()

def random_baseline(uid, data, k=100):
    pool = [m for m in popular_items[:5000] if m not in data["all_watched"]]
    random.seed(uid)
    random.shuffle(pool)
    return pool[:k]

def popular_baseline(uid, data, k=100):
    return [m for m in popular_items if m not in data["all_watched"]][:k]

print("Running Random baseline...")
random_results = evaluate_method(random_baseline, test_set, method_name="Random")

print("Running MostPopular baseline...")
popular_results = evaluate_method(popular_baseline, test_set, method_name="Popular")


Running Random baseline...
  Random: done (979 users, 3.3s)
Running MostPopular baseline...
  Popular: done (979 users, 6.8s)


### ItemKNN Baseline
Item-based CF using co-occurrence similarity

In [8]:
import faiss
import time
import numpy as np
from scipy.sparse import lil_matrix, csr_matrix
from collections import defaultdict

print("Building ItemKNN model from train_ratings using GPU...")
t0 = time.time()

# Filter positive interactions
pos_train = train_ratings[train_ratings["rating"] >= 3.5]
all_items = sorted(set(train_ratings["movieId"].unique()))
item_to_idx = {m: i for i, m in enumerate(all_items)}
n_items = len(all_items)
print(f"  Items: {n_items:,}")

# Build Item Vectors (Latent Profiles)
user_list = sorted(pos_train["userId"].unique())
user_to_idx = {u: i for i, u in enumerate(user_list)}
n_users = len(user_list)

# Construct sparse matrix: rows=items, cols=users
item_user_matrix = lil_matrix((n_items, n_users), dtype=np.float32)
for uid, mid in zip(pos_train["userId"].values, pos_train["movieId"].values):
    item_user_matrix[item_to_idx[mid], user_to_idx[uid]] = 1.0

item_vectors = item_user_matrix.toarray().astype('float32')

# GPU Cosine Similarity via FAISS
print("  Computing item similarity on A100 GPU...")
res = faiss.StandardGpuResources()
dim = n_users

# Normalize vectors to unit length so Inner Product = Cosine Similarity
faiss.normalize_L2(item_vectors)

# Setup GPU Index
index_flat = faiss.IndexFlatIP(dim)
gpu_index = faiss.index_cpu_to_gpu(res, 0, index_flat)
gpu_index.add(item_vectors)

# Search for top 51 (50 neighbors + the item itself)
KNN_NEIGHBORS = 50
sims, indices = gpu_index.search(item_vectors, KNN_NEIGHBORS + 1)

# Map results back to item_neighbors dictionary
item_neighbors = {}
for i in range(n_items):
    # Skip the first result because it's the item itself (sim=1.0)
    neighbor_list = []
    for j in range(1, KNN_NEIGHBORS + 1):
        if sims[i, j] > 0:
            neighbor_list.append((int(indices[i, j]), float(sims[i, j])))
    item_neighbors[i] = neighbor_list

print(f"  ItemKNN built in {time.time()-t0:.1f}s")

def itemknn_baseline(uid, data, k=100):
    user_liked = [item_to_idx[m] for m in data["history"][data["history"]["rating"]>=3.5]["movieId"]
                  if m in item_to_idx]
    if not user_liked:
        return popular_baseline(uid, data, k)

    scores = defaultdict(float)
    # Use last 50 liked items to speed up inference
    for liked_idx in user_liked[-50:]:
        for nb_idx, sim in item_neighbors.get(liked_idx, []):
            mid = all_items[nb_idx]
            if mid not in data["history_mids"]:
                scores[mid] += sim

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [mid for mid, _ in ranked[:k]]

print("Running ItemKNN baseline...")
itemknn_results = evaluate_method(itemknn_baseline, test_set, method_name="ItemKNN")

Building ItemKNN model from train_ratings using GPU...
  Items: 84,429
  Computing item similarity on A100 GPU...
  ItemKNN built in 334.2s
Running ItemKNN baseline...
  ItemKNN: done (979 users, 2.8s)


In [9]:
print(itemknn_results)

{10: {'NDCG': np.float64(0.08901384816845352), 'Hit': np.float64(0.45352400408580185), 'MRR': np.float64(0.23943468662636097), 'ILD': np.float64(0.7676228649092115), 'CCDR': np.float64(0.009601634320735446), 'Coverage': 0.017719929211622996}, 20: {'NDCG': np.float64(0.11510412672727093), 'Hit': np.float64(0.6016343207354443), 'MRR': np.float64(0.23943468662636097), 'ILD': np.float64(0.7823044933938165), 'CCDR': np.float64(0.013176710929519919), 'Coverage': 0.017719929211622996}, 30: {'NDCG': np.float64(0.13449634026605062), 'Hit': np.float64(0.686414708886619), 'MRR': np.float64(0.23943468662636097), 'ILD': np.float64(0.7909121686003444), 'CCDR': np.float64(0.01675178753830439), 'Coverage': 0.017719929211622996}}


In [16]:
import gc
import torch

for var in ['item_vectors', 'item_user_matrix', 'sims', 'indices', 'gpu_index']:
    if var in globals():
        del globals()[var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Memory cleaned up.')

Memory cleaned up.


### LightGCN Baselines

- **LightGCN**: Light Graph Convolution (He et al., 2020)

Both trained on the same `train_ratings.csv` as XSimGCL

In [17]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'bool_'):
    np.bool_ = bool
if not hasattr(np, 'int_'):
    np.int_ = np.int64
if not hasattr(np, 'complex_'):
    np.complex_ = complex
if not hasattr(np, 'object_'):
    np.object_ = object
if not hasattr(np, 'unicode_'):
    np.unicode_ = np.str_


In [18]:
import tempfile, shutil
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.trainer import Trainer
from recbole.utils import init_seed
import recbole.utils

def prepare_recbole_data():
    """Write RecBole .inter file from our train split (same as XSimGCL)."""
    tmpdir = Path(tempfile.mkdtemp())
    ds_name = "ml32m_eval"
    ds_dir = tmpdir / ds_name
    ds_dir.mkdir()

    # Use train_ratings (positive interactions only, rating >= 3.5)
    pos = train_ratings[train_ratings["rating"] >= 3.5][["userId","movieId","timestamp"]].copy()
    pos.columns = ["user_id:token", "item_id:token", "timestamp:float"]
    pos.to_csv(ds_dir / f"{ds_name}.inter", sep="\t", index=False)
    print(f"  RecBole data: {len(pos):,} interactions → {ds_dir}")
    return tmpdir, ds_name


def run_recbole_model(model_name, tmpdir, ds_name, epochs=10, emb_size=128):
    config_dict = {
        "model": model_name,
        "dataset": ds_name,
        "data_path": str(tmpdir),
        "USER_ID_FIELD": "user_id",
        "ITEM_ID_FIELD": "item_id",
        "TIME_FIELD": "timestamp",
        "load_col": {"inter": ["user_id", "item_id", "timestamp"]},
        "eval_args": {
            "split": {"RS": [0.9, 0.05, 0.05]},
            "order": "TO",
            "mode": {"valid": "full", "test": "full"},
        },
        "training_neg_sample_num": 1,
        "epochs": 100,


        "train_batch_size": 32768,
        "eval_batch_size": 8192,

        "embedding_size": emb_size,
        "learning_rate": 0.001,
        "metrics": ["NDCG", "Hit"],
        "topk": [10, 20, 30],
        "valid_metric": "NDCG@20",
        "show_progress": True,
        "device": device,
    }

    config = Config(model=model_name, dataset=ds_name, config_dict=config_dict)
    init_seed(config["seed"], config["reproducibility"])
    dataset = create_dataset(config)
    uid_field = dataset.uid_field
    iid_field = dataset.iid_field
    ext2int_user = dataset.field2token_id[uid_field]
    ext2int_item = dataset.field2token_id[iid_field]
    int2ext_item = {v: k for k, v in ext2int_item.items()}
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model = recbole.utils.get_model(model_name)(config, train_data.dataset).to(config["device"])
    trainer = Trainer(config, model)
    trainer.fit(train_data, valid_data)
    return model, dataset, ext2int_user, ext2int_item, int2ext_item, config

tmpdir, ds_name = prepare_recbole_data()

  RecBole data: 20,218,546 interactions → /tmp/tmp68cpjlbr/ml32m_eval


In [13]:
!pip install kmeans_pytorch

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'bool_'):
    np.bool_ = bool
if not hasattr(np, 'int_'):
    np.int_ = np.int64
if not hasattr(np, 'complex_'):
    np.complex_ = complex
if not hasattr(np, 'object_'):
    np.object_ = object
if not hasattr(np, 'unicode_'):
    np.unicode_ = np.str_

import gc
import torch

for var in ['item_vectors', 'item_user_matrix', 'sims', 'indices', 'gpu_index']:
    if var in globals():
        del globals()[var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Memory cleaned up.')

In [ ]:
print("\n" + "="*60)
print("  Training LightGCN")
print("="*60)
lgcn_model, lgcn_ds, lgcn_u2i, lgcn_i2e, lgcn_i2ext, lgcn_cfg = \
    run_recbole_model("LightGCN", tmpdir, ds_name, epochs=10, emb_size=128)

def lightgcn_recommend(uid, data, k=100):
    uid_str = str(uid)
    if uid_str not in lgcn_u2i:
        return popular_baseline(uid, data, k)
    int_uid = lgcn_u2i[uid_str]
    uid_tensor = torch.tensor([int_uid]).to(lgcn_cfg["device"])
    with torch.no_grad():
        scores = lgcn_model.full_sort_predict(uid_tensor.unsqueeze(0))
    scores = scores.cpu().numpy().flatten()
    top_indices = np.argsort(scores)[::-1]
    result = []
    for idx in top_indices:
        ext_item = lgcn_i2ext.get(int(idx))
        if ext_item and ext_item != "[PAD]":
            mid = int(ext_item)
            if mid not in data["history_mids"]:
                result.append(mid)
                if len(result) >= k: break
    return result

print("Running LightGCN evaluation...")
lightgcn_results = evaluate_method(lightgcn_recommend, test_set, method_name="LightGCN")

In [ ]:
print(lightgcn_results)

## CineMatch Variants
- **Content-Only**: FAISS retrieval only (α=1, β=0)
- **CF-Only**: XSimGCL only (α=0, β=1)
- **CineMatch Full**: Fused with heuristic gating (α≈0.6, β≈0.3)
- **CineMatch + DPP**: Full pipeline with DPP diversity selection

In [ ]:
import numpy as np
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'bool_'):
    np.bool_ = bool
if not hasattr(np, 'int_'):
    np.int_ = np.int64
if not hasattr(np, 'complex_'):
    np.complex_ = complex
if not hasattr(np, 'object_'):
    np.object_ = object
if not hasattr(np, 'unicode_'):
    np.unicode_ = np.str_

import gc
import torch

for var in ['item_vectors', 'item_user_matrix', 'sims', 'indices', 'gpu_index']:
    if var in globals():
        del globals()[var]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Memory cleaned up.')

In [ ]:
def faiss_retrieve_mids(query_vec, index, k=500):
    """Retrieve movieIds from FAISS index (TMDB indices only)."""
    vec = query_vec.astype("float32").reshape(1, -1)
    norm = np.linalg.norm(vec)
    if norm > 0: vec = vec / norm
    scores, ids = index.search(vec, k)
    results = []
    seen = set()
    for s, fid in zip(scores[0], ids[0]):
        if fid < 0 or fid in seen: continue
        seen.add(int(fid))
        mid = tmdb_to_ml.get(int(fid))
        if mid:
            results.append((int(mid), float(s)))
    return results

def xsimgcl_retrieve(uid, k=300, exclude=None):
    if uid not in user_id_map:
        return []
    u_vec = user_emb[user_id_map[uid]]
    scores = item_emb @ u_vec
    exclude = exclude or set()
    results = []
    for idx in np.argsort(scores)[::-1]:
        if len(results) >= k: break
        mid = idx_to_movieid.get(int(idx))
        if mid and mid not in exclude:
            results.append((mid, float(scores[idx])))
    return results

def build_query_from_history(history_df, k_recent=5):
    recent = history_df[history_df["rating"] >= 3.5].sort_values("timestamp", ascending=False).head(k_recent)
    parts = []
    for _, r in recent.iterrows():
        mid = int(r["movieId"])
        g = movie_genres.get(mid, "")
        title = movies[movies["movieId"]==mid]["title"].values
        if len(title) > 0:
            parts.append(f"'{title[0]}' ({g[:40]})")
    if not parts: return "popular movies"
    return "Instruct: Given a movie description, retrieve semantically similar movies.\nQuery: " + ", ".join(parts)

def minmax(arr):
    mn, mx = arr.min(), arr.max()
    return np.ones_like(arr)*0.5 if mx-mn<1e-10 else (arr-mn)/(mx-mn)

def cinematch_recommend(uid, data, index_name="tmdb_bge", alpha=0.6, beta=0.3,
                        use_dpp=False, k=100):
    if index_name not in faiss_indices:
        return popular_baseline(uid, data, k)

    index = faiss_indices[index_name]
    query_text = build_query_from_history(data["history"])
    with torch.no_grad():
        q_vec = bge_model.encode([query_text], normalize_embeddings=True,
                                 convert_to_numpy=True, show_progress_bar=False).astype("float32")
    if q_vec.shape[1] < index.d:
        q_vec = np.pad(q_vec, ((0,0),(0,index.d-q_vec.shape[1])))
    elif q_vec.shape[1] > index.d:
        q_vec = q_vec[:, :index.d]

    # FAISS retrieval
    faiss_results = faiss_retrieve_mids(q_vec[0], index, k=500)

    # CF retrieval
    cf_results = []
    if beta > 0.01:
        cf_results = xsimgcl_retrieve(uid, k=300, exclude=data["history_mids"])

    # Fuse
    candidates = {}
    for mid, score in faiss_results:
        if mid not in data["history_mids"]:
            candidates[mid] = {"faiss": score, "cf": 0.0}
    for mid, score in cf_results:
        if mid in candidates:
            candidates[mid]["cf"] = score
        elif mid not in data["history_mids"]:
            candidates[mid] = {"faiss": 0.0, "cf": score}

    if not candidates:
        return popular_baseline(uid, data, k)

    mids = list(candidates.keys())
    faiss_s = np.array([candidates[m]["faiss"] for m in mids])
    cf_s = np.array([candidates[m]["cf"] for m in mids])
    fused = alpha * minmax(faiss_s) + beta * minmax(cf_s)
    ranked = sorted(zip(mids, fused), key=lambda x: x[1], reverse=True)

    if use_dpp and len(ranked) > k:
        pool = ranked[:min(150, len(ranked))]
        n = len(pool)
        q_vals = np.array([max(s, 0.01)**0.5 for _, s in pool])
        genre_sets = [set(movie_genres.get(m, "").split("|")) for m, _ in pool]
        sim = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                union = genre_sets[i] | genre_sets[j]
                inter = genre_sets[i] & genre_sets[j]
                sim[i,j] = len(inter)/max(len(union),1)
        L = np.outer(q_vals, q_vals) * sim
        selected, remaining = [], list(range(n))
        for _ in range(min(k, n)):
            if not remaining: break
            best, best_gain = None, -1e30
            if not selected:
                for i in remaining:
                    if L[i,i] > best_gain: best_gain = L[i,i]; best = i
            else:
                sel = np.array(selected)
                det_cur = max(np.linalg.det(L[np.ix_(sel, sel)]), 1e-30)
                for i in remaining:
                    ns = np.append(sel, i)
                    gain = np.linalg.det(L[np.ix_(ns, ns)]) / det_cur
                    if gain > best_gain: best_gain = gain; best = i
            if best is not None:
                selected.append(best); remaining.remove(best)
        return [pool[i][0] for i in selected]

    return [mid for mid, _ in ranked[:k]]

In [ ]:
cinematch_all_results = {}

for idx_name in faiss_indices:
    if "imdb" in idx_name:
        continue  # IMDB FAISS cannot map back to ML movieIds for held-out eval

    print(f"\n{'═'*60}")
    print(f"  Evaluating CineMatch on: {idx_name}")
    print(f"{'═'*60}")

    # Content-Only (α=1, β=0)
    def content_only(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=1.0, beta=0.0, k=k)
    r = evaluate_method(content_only, test_set, method_name=f"{idx_name} Content-Only")
    cinematch_all_results[f"{idx_name}_content"] = r

    # CF-Only (α=0, β=1)
    def cf_only(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=0.0, beta=1.0, k=k)
    r = evaluate_method(cf_only, test_set, method_name=f"{idx_name} CF-Only")
    cinematch_all_results[f"{idx_name}_cf"] = r

    # Full Fusion (α=0.6, β=0.3)
    def full_fusion(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=0.6, beta=0.3, k=k)
    r = evaluate_method(full_fusion, test_set, method_name=f"{idx_name} Full")
    cinematch_all_results[f"{idx_name}_full"] = r

    # Full + DPP
    def full_dpp(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=0.6, beta=0.3,
                                   use_dpp=True, k=k)
    r = evaluate_method(full_dpp, test_set, method_name=f"{idx_name} Full+DPP")
    cinematch_all_results[f"{idx_name}_full_dpp"] = r

## Results

In [ ]:
K = 20  # Primary evaluation cutoff

rows = []
for name, res in [("Random", random_results),
                  ("MostPopular", popular_results),
                  ("ItemKNN", itemknn_results),
                  ("BPR-MF", bpr_results),
                  ("LightGCN (SOTA)", lightgcn_results)]:
    if K in res:
        rows.append({"Method": name, **{k: f"{v:.4f}" for k, v in res[K].items()}})

for key, res in cinematch_all_results.items():
    if K in res:
        label = key.replace("_", " ").title()
        rows.append({"Method": f"CineMatch {label}", **{k: f"{v:.4f}" for k, v in res[K].items()}})

results_df = pd.DataFrame(rows)
print(f"\n{'═'*80}")
print(f"  EVALUATION RESULTS @ K={K}")
print(f"  Test Users: {len(test_set)} | Holdout: {split_meta['n_holdout_per_user']} per user")
print(f"  Split seed: {split_meta['eval_seed']} | Train: {split_meta['total_train_ratings']:,} ratings")
print(f"{'═'*80}\n")
display(results_df)

# Also K=10 and K=30
for k_val in [10, 30]:
    rows_k = []
    for name, res in [("Random", random_results), ("MostPopular", popular_results),
                      ("ItemKNN", itemknn_results), ("BPR-MF", bpr_results),
                      ("LightGCN", lightgcn_results)]:
        if k_val in res:
            rows_k.append({"Method": name, **{k: f"{v:.4f}" for k, v in res[k_val].items()}})
    for key, res in cinematch_all_results.items():
        if k_val in res:
            rows_k.append({"Method": key, **{k: f"{v:.4f}" for k, v in res[k_val].items()}})
    print(f"\n@ K={k_val}:")
    display(pd.DataFrame(rows_k))

In [ ]:
print(f"\n XSimGCL on ML-32M: NDCG@20 = {manifest['results'].get('test_result', 'N/A')}")

## Cold-Start vs Warm-Start Analysis
Split test users by activity level. Validates the MLP gating rationale:
- Cold users → Content (FAISS) should dominate
- Power users → CF (XSimGCL) should dominate
- Fusion should help across the board

In [ ]:
cold_users = {uid: d for uid, d in test_set.items() if len(d["history"]) < 20}
warm_users = {uid: d for uid, d in test_set.items() if 50 <= len(d["history"]) < 100}
power_users = {uid: d for uid, d in test_set.items() if len(d["history"]) >= 100}
print(f"Cold (<20 ratings): {len(cold_users)}")
print(f"Warm (20-100):      {len(warm_users)}")
print(f"Power (100+):       {len(power_users)}")

idx_name = "tmdb_bge" if "tmdb_bge" in faiss_indices else list(faiss_indices.keys())[0]

for seg_name, seg in [("Cold", cold_users), ("Warm", warm_users), ("Power", power_users)]:
    if not seg:
        print(f"  {seg_name}: no users, skipping"); continue
    print(f"\n{'─'*40} {seg_name} Users {'─'*40}")

    def content_fn(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=1.0, beta=0.0, k=k)
    def cf_fn(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=0.0, beta=1.0, k=k)
    def full_fn(uid, data, k=100, _idx=idx_name):
        return cinematch_recommend(uid, data, index_name=_idx, alpha=0.6, beta=0.3, k=k)

    c_res = evaluate_method(content_fn, seg, method_name=f"{seg_name}-Content")
    f_res = evaluate_method(cf_fn, seg, method_name=f"{seg_name}-CF")
    u_res = evaluate_method(full_fn, seg, method_name=f"{seg_name}-Full")

    seg_rows = []
    for label, r in [(f"{seg_name} Content-Only", c_res),
                     (f"{seg_name} CF-Only", f_res),
                     (f"{seg_name} CineMatch Full", u_res)]:
        if 20 in r:
            seg_rows.append({"Method": label, **{k: f"{v:.4f}" for k, v in r[20].items()}})
    display(pd.DataFrame(seg_rows))

## DPP Diversity Ablation

In [ ]:
idx_name = "tmdb_bge" if "tmdb_bge" in faiss_indices else list(faiss_indices.keys())[0]

def no_dpp(uid, data, k=30):
    return cinematch_recommend(uid, data, index_name=idx_name, alpha=0.6, beta=0.3,
                               use_dpp=False, k=k)
def with_dpp(uid, data, k=30):
    return cinematch_recommend(uid, data, index_name=idx_name, alpha=0.6, beta=0.3,
                               use_dpp=True, k=k)

print("Without DPP:")
no_dpp_res = evaluate_method(no_dpp, test_set, k_values=[30], method_name="No DPP")
print("With DPP:")
dpp_res = evaluate_method(with_dpp, test_set, k_values=[30], method_name="With DPP")

dpp_comparison = []
for label, r in [("Without DPP", no_dpp_res), ("With DPP", dpp_res)]:
    if 30 in r:
        dpp_comparison.append({"Method": label, **{k: f"{v:.4f}" for k, v in r[30].items()}})
print("\nDPP Ablation @ K=30:")
display(pd.DataFrame(dpp_comparison))
if 30 in no_dpp_res and 30 in dpp_res:
    ild_gain = dpp_res[30]["ILD"] - no_dpp_res[30]["ILD"]
    ndcg_cost = no_dpp_res[30]["NDCG"] - dpp_res[30]["NDCG"]
    print(f"\nILD gain from DPP:  +{ild_gain:.4f}")
    print(f"NDCG trade-off:     -{ndcg_cost:.4f}")
    print(f"Diversity/Quality ratio: {abs(ild_gain/max(ndcg_cost,1e-6)):.2f}x")

## Summary & Takeaways

In [ ]:
print("═" * 70)
print("  CineMatch Evaluation Summary")
print("═" * 70)
print(f"  Train/Test split from: 7)XSimGCL_Train.ipynb (seed={split_meta['eval_seed']})")
print(f"  Train ratings: {split_meta['total_train_ratings']:,}")
print(f"  Test users: {split_meta['n_test_users']} | Holdout/user: {split_meta['n_holdout_per_user']}")
print(f"  Activity distribution: {split_meta.get('activity_distribution', {})}")
print(f"  FAISS indices evaluated: {[k for k in faiss_indices if 'imdb' not in k]}")
print()
print("  Baselines:")
print("    • Random  • MostPopular  • ItemKNN  • BPR-MF  • LightGCN")
print()
print("  CineMatch variants:")
print("    • Content-Only (α=1)  • CF-Only (β=1)")
print("    • Full Fusion (α=0.6, β=0.3)  • Full + DPP")
print()
print("  Key findings to report:")
print("    1. Fusion > Content-Only AND CF-Only → proves late fusion adds value")
print("    2. Content dominates for cold users → validates α-heavy gating")
print("    3. CF dominates for power users → validates β-heavy gating")
print("    4. DPP improves ILD with minimal NDCG cost → proves diversity value")
print("    5. Compare vs LightGCN/BPR → shows SOTA competitiveness")
print("═" * 70)